<a href="https://colab.research.google.com/github/Isabela-Tellez/BootcampIA/blob/main/05.%20Mayo-04/tarea_pydantic_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Actividad: Validación con Pydantic

Contexto: Habéis programado el esquema UserCreate y un cliente Frontend os hace una petición POST enviando este JSON en el cuerpo del mensaje:

from pydantic import BaseModel, EmailStr

Esquema de Entrada (Lo que el cliente debe enviar en el POST)
class UserCreate(BaseModel):
    username: str
    email: EmailStr
    age: int
    is_active: bool = True

{
    "username": "admin_f5",
    "email": "usuario.f5@com",
    "age": "25",
    "is_active": "yes"
}

Misión: Analizad el JSON basándoos en el esquema Pydantic.

• ¿Qué responderá la API?
• ¿Se guardará en la base de datos o fallará?
• Enumerad los posibles problemas.

Aquí tienes el análisis detallado de la validación con Pydantic:

━━━━━━━━━━━━━━━━━━

Respuesta de la API
La API fallará y devolverá un error de validación HTTP 422 (Unprocessable Entity) con un mensaje similar a:

{
"detail": [
{
"loc": ["body", "email"],
"msg": "value is not a valid email address",
"type": "value_error.email"
},
{
"loc": ["body", "age"],
"msg": "value is not a valid integer",
"type": "type_error.integer"
},
{
"loc": ["body", "is_active"],
"msg": "value could not be parsed to a boolean",
"type": "type_error.bool"
}
]
}

━━━━━━━━━━━━━━━━━━

¿Se guardará en la base de datos?
No. Pydantic bloqueará la operación antes de que llegue al ORM o a la base de datos. La validación ocurre en el middleware de FastAPI, que usa Pydantic para parsear y validar los datos entrantes.

━━━━━━━━━━━━━━━━━━

Problemas Encontrados
1. Email inválido:
- El campo email usa EmailStr, que exige un formato válido (ej: usuario@dominio.com).
- Error: "usuario.f5@com" falta el dominio completo (debería ser @compania.com o similar).

2. Tipo incorrecto en age:
- El esquema espera un int, pero el JSON envía un string ("25").
- Pydantic no hace casting automático de strings a enteros.

3. Valor no booleano en is_active:
- El esquema define is_active: bool, pero el JSON envía "yes" (string).
- Valores válidos para bool: true, false, 1, 0 (en JSON).

4. Campo opcional ignorado:
- Aunque is_active tiene un valor por defecto (True), el cliente envió un valor inválido, lo que anula el default y fuerza la validación.

━━━━━━━━━━━━━━━━━━

Solución Recomendada
1. Corregir el JSON del cliente:
{
"username": "admin_f5",
"email": "usuario.f5@compania.com",
"age": 25,
"is_active": true
}

2. Opcional: Si el frontend no puede garantizar tipos correctos, usar Union en Pydantic para aceptar múltiples formatos:
age: Union[int, str]  # Requiere manejo adicional en el backend.

¿Necesitas que profundice en cómo FastAPI integra Pydantic o en cómo personalizar los mensajes de error?

Perfecto. Vamos a profundizar en la "sala de máquinas" de FastAPI para entender cómo exprime a Pydantic y cómo puedes tomar el control total de las respuestas cuando algo sale mal.

━━━━━━━━━━━━━━━━━━

1. La Integración: El "Apretón de Manos" Automático
Cuando defines un modelo de Pydantic como parámetro en una función de FastAPI, ocurre una secuencia automática de eventos:

1. Extracción: FastAPI lee el Content-Type de la petición. Si es application/json, extrae el cuerpo.
2. Validación/Parsing: FastAPI le pasa ese diccionario a Pydantic. Pydantic intenta "encajar" los datos en el molde (modelo).
3. Inyección de Dependencia: Si la validación es exitosa, FastAPI te entrega el objeto ya instanciado y tipado (con autocompletado en tu IDE).
4. Cortafuegos (Error 422): Si falla, FastAPI detiene la ejecución de tu función y lanza una excepción RequestValidationError.

━━━━━━━━━━━━━━━━━━

2. Personalización de Errores (Override del 422)
Por defecto, FastAPI devuelve un JSON muy estructurado pero a veces demasiado técnico para el usuario final. Puedes capturar este error globalmente para devolver mensajes más amigables:

from fastapi import FastAPI, Request, status
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse

app = FastAPI()

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    # Aquí puedes loguear el error o transformarlo
    errors = []
    for error in exc.errors():
        # Personalizamos el mensaje según el tipo de error
        field = error['loc'][-1]
        msg = f"El campo '{field}' tiene un problema: {error['msg']}"
        errors.append({"error": msg})

    return JSONResponse(
        status_code=status.HTTP_400_BAD_REQUEST, # Cambiamos de 422 a 400 si preferimos
        content={"status": "error", "detalles": errors},
    )

━━━━━━━━━━━━━━━━━━

3. Validación Avanzada con Field y @field_validator
Pydantic v2 (el estándar actual) permite añadir reglas de negocio directamente en el esquema, descargando de trabajo a tu lógica de base de datos.

Uso de Field:
Sirve para metadatos y validaciones simples (rangos, longitud).
from pydantic import BaseModel, Field, EmailStr

class UserCreate(BaseModel):
    username: str = Field(..., min_length=3, max_length=20, pattern="^[a-zA-Z0-9_]+$")
    age: int = Field(..., gt=0, lt=120) # Debe ser > 0 y < 120
    email: EmailStr

Uso de @field_validator:
Para lógica compleja que dependa de comparaciones o reglas externas.
from pydantic import field_validator

class UserCreate(BaseModel):
    # ... campos anteriores ...

    @field_validator('username')
    @classmethod
    def prevent_admin_impersonation(cls, v: str) -> str:
        if "admin" in v.lower():
            raise ValueError("No puedes usar la palabra 'admin' en el nombre de usuario")
        return v

━━━━━━━━━━━━━━━━━━

4. ¿Por qué esto es "Arquitectura de Élite"?
• Separación de Responsabilidades: Tu lógica de negocio (el servicio que guarda en la DB) nunca recibe datos "sucios". Solo recibe objetos válidos.
• Documentación Viva: Todo lo que pongas en Field o en el modelo se refleja automáticamente en /docs (Swagger UI). Si pones que la edad debe ser > 0, el Swagger lo mostrará como una restricción técnica.
• Seguridad (Type Safety): Al usar is_active: bool, Pydantic se asegura de que nadie inyecte un string malicioso donde se espera un booleano.

¿Te gustaría ver cómo manejar esto cuando los datos vienen de un formulario HTML en lugar de un JSON, o prefieres que pasemos a la parte de Base de Datos con SQLAlchemy?